In [10]:
def bitswap_postprocess(
    layout_yaml,
    candidate_locations,
    wake_loss_matrix,
    radius_m,
    min_distance=396.0,
    top_per_turbine=5,
    top_global=100,
    max_iterations=100,
    min_improvement=1e-9,
    match_tol=1e-6,
    verbose=True,
):
    """
    Hybrid 1-bit-swap local search.

    Screening:
        pairwise QUBO objective.

    Acceptance:
        full physical AEP.

    Per ogni iterazione:
        1. genera gli swap feasible entro radius_m;
        2. tiene le top-N mosse QUBO per ogni turbina;
        3. tiene le top-N mosse globalmente;
        4. calcola il full AEP solo su queste;
        5. accetta il miglior swap se migliora il vero AEP.
    """

    candidate_locations = np.asarray(
        candidate_locations,
        dtype=float,
    )

    L = np.asarray(
        wake_loss_matrix,
        dtype=float,
    )

    n_candidates = len(candidate_locations)

    if L.shape != (n_candidates, n_candidates):
        raise ValueError(
            "wake_loss_matrix deve avere shape "
            f"({n_candidates}, {n_candidates}), "
            f"ricevuta {L.shape}."
        )

    # --------------------------------------------------
    # 1. Spatial index
    # --------------------------------------------------

    tree = cKDTree(candidate_locations)

    # --------------------------------------------------
    # 2. Invalid spacing pairs
    # --------------------------------------------------

    invalid_pairs = tree.query_pairs(
        r=min_distance,
        output_type="ndarray",
    )

    if len(invalid_pairs) > 0:

        distances = np.linalg.norm(
            candidate_locations[invalid_pairs[:, 0]]
            - candidate_locations[invalid_pairs[:, 1]],
            axis=1,
        )

        invalid_pairs = invalid_pairs[
            distances < min_distance
        ]

    invalid_pairs = np.asarray(
        invalid_pairs,
        dtype=np.int32,
    )

    # conflict graph
    conflicts = [
        set()
        for _ in range(n_candidates)
    ]

    for i, j in invalid_pairs:
        conflicts[i].add(j)
        conflicts[j].add(i)

    # --------------------------------------------------
    # 3. Load initial layout
    # --------------------------------------------------

    initial_positions = np.asarray(
        load_layout_coordinates(layout_yaml),
        dtype=float,
    )

    n_turbines = len(initial_positions)

    # --------------------------------------------------
    # 4. YAML positions -> candidate indices
    # --------------------------------------------------

    distances, selected_indices = tree.query(
        initial_positions,
        k=1,
    )

    if np.any(distances > match_tol):
        raise ValueError(
            "Alcune turbine del layout YAML non sono "
            "presenti in candidate_locations. "
            f"Max distance = {distances.max():.6f} m"
        )

    if len(np.unique(selected_indices)) != n_turbines:
        raise ValueError(
            "Più turbine sono state mappate sulla "
            "stessa candidate position."
        )

    selected = np.zeros(
        n_candidates,
        dtype=bool,
    )

    selected[selected_indices] = True

    # --------------------------------------------------
    # 5. Validate initial spacing
    # --------------------------------------------------

    if len(invalid_pairs) > 0:

        violations = (
            selected[invalid_pairs[:, 0]]
            & selected[invalid_pairs[:, 1]]
        )

        if np.any(violations):
            raise ValueError(
                "Il layout iniziale viola lo spacing."
            )

    # --------------------------------------------------
    # 6. Initial full AEP
    # --------------------------------------------------

    current_positions = candidate_locations[
        np.flatnonzero(selected)
    ]

    current_aep = compute_aep_from_coords(
        current_positions
    )

    initial_aep = current_aep

    history = []

    if verbose:
        print(f"Candidates: {n_candidates}")
        print(f"Turbines: {n_turbines}")
        print(f"Initial AEP: {current_aep:.6f}")

    # --------------------------------------------------
    # 7. Iterative bit-swap
    # --------------------------------------------------

    for iteration in range(max_iterations):

        current_selected = np.flatnonzero(
            selected
        )

        shortlist = []

        # ==============================================
        # Stage 1:
        # Top-N QUBO swaps PER TURBINE
        # ==============================================

        for i in current_selected:

            others = current_selected[
                current_selected != i
            ]

            # Wake contribution della posizione corrente
            current_qubo_cost = L[
                i,
                others,
            ].sum()

            local_candidates = tree.query_ball_point(
                candidate_locations[i],
                r=radius_m,
            )

            turbine_moves = []

            for j in local_candidates:

                if selected[j]:
                    continue

                # --------------------------------------
                # Spacing feasibility
                # --------------------------------------

                spacing_ok = True

                for k in conflicts[j]:

                    # i viene rimossa dallo swap
                    if selected[k] and k != i:
                        spacing_ok = False
                        break

                if not spacing_ok:
                    continue

                # --------------------------------------
                # Exact delta del pairwise QUBO
                #
                # ΔH < 0 -> promettente
                # --------------------------------------

                new_qubo_cost = L[
                    j,
                    others,
                ].sum()

                delta_qubo = (
                    new_qubo_cost
                    - current_qubo_cost
                )

                turbine_moves.append(
                    (
                        float(delta_qubo),
                        int(i),
                        int(j),
                    )
                )

            # migliori candidate per questa turbina
            turbine_moves.sort(
                key=lambda x: x[0]
            )

            shortlist.extend(
                turbine_moves[
                    :top_per_turbine
                ]
            )

        # ==============================================
        # Stage 2:
        # Top-N GLOBAL QUBO swaps
        # ==============================================

        if not shortlist:

            if verbose:
                print(
                    "Nessuno swap feasible trovato."
                )

            break

        shortlist.sort(
            key=lambda x: x[0]
        )

        shortlist = shortlist[
            :top_global
        ]

        if verbose:
            print(
                f"\nIteration {iteration + 1}: "
                f"{len(shortlist)} swaps "
                f"selected for full-AEP evaluation"
            )

        # ==============================================
        # Stage 3:
        # Full physics evaluation
        # ==============================================

        best_aep = current_aep
        best_swap = None
        best_delta_qubo = None

        for delta_qubo, i, j in shortlist:

            # apply temporary swap
            selected[i] = False
            selected[j] = True

            trial_positions = candidate_locations[
                np.flatnonzero(selected)
            ]

            trial_aep = compute_aep_from_coords(
                trial_positions
            )

            # restore
            selected[j] = False
            selected[i] = True

            if (
                trial_aep
                > best_aep + min_improvement
            ):
                best_aep = trial_aep
                best_swap = (i, j)
                best_delta_qubo = delta_qubo

        # ==============================================
        # No real AEP improvement
        # ==============================================

        if best_swap is None:

            if verbose:
                print(
                    "Nessuno degli swap shortlist "
                    "migliora il full AEP."
                )

            break

        # ==============================================
        # Accept best real swap
        # ==============================================

        i, j = best_swap

        selected[i] = False
        selected[j] = True

        improvement = (
            best_aep - current_aep
        )

        history.append({
            "iteration": iteration + 1,

            "from_index": int(i),
            "to_index": int(j),

            "from_position":
                candidate_locations[i].copy(),

            "to_position":
                candidate_locations[j].copy(),

            "delta_qubo":
                float(best_delta_qubo),

            "delta_aep":
                float(improvement),

            "aep":
                float(best_aep),
        })

        current_aep = best_aep

        if verbose:
            print(
                f"Accepted: "
                f"{i} -> {j}"
            )

            print(
                f"ΔQUBO = "
                f"{best_delta_qubo:.6f}"
            )

            print(
                f"ΔAEP  = "
                f"+{improvement:.6f}"
            )

            print(
                f"AEP   = "
                f"{current_aep:.6f}"
            )

    # --------------------------------------------------
    # 8. Final result
    # --------------------------------------------------

    final_indices = np.flatnonzero(
        selected
    )

    final_positions = candidate_locations[
        final_indices
    ]

    valid_cardinality = (
        len(final_indices) == n_turbines
    )

    if len(invalid_pairs) > 0:

        spacing_violations = (
            selected[invalid_pairs[:, 0]]
            & selected[invalid_pairs[:, 1]]
        )

        valid_spacing = not np.any(
            spacing_violations
        )

    else:
        valid_spacing = True

    return {
        "selected_indices":
            final_indices,

        "selected_locations":
            final_positions,

        "initial_aep":
            float(initial_aep),

        "final_aep":
            float(current_aep),

        "improvement_aep":
            float(
                current_aep - initial_aep
            ),

        "n_swaps":
            len(history),

        "valid_cardinality":
            valid_cardinality,

        "valid_spacing":
            valid_spacing,

        "history":
            history,
    }


In [ ]:
import sys
import numpy as np
from scipy.spatial import cKDTree

sys.path.append("../..")
from src.qubo_windfarm_layout.model import get_farm_area, get_mask, build_qubo_from_wake_matrix_optimized, build_wake_loss_matrix_optimized, build_wake_loss_matrix_optimized, load_wake_loss_data
from src.qubo_windfarm_layout.evaluation import load_layout_coordinates, get_farm_area
from src.qubo_windfarm_layout.model import get_farm_area, compute_aep_from_coords

# Variables
GRID_RESOLUTION = 128 # m
MIN_DISTANCE = 396  # 2 * 198 m


# Get candidates
polygons, farm_area = get_farm_area()
X, Y, mask = get_mask(farm_area=farm_area, grid_resolution=GRID_RESOLUTION)

candidate_locations = np.column_stack([
    X[mask],
    Y[mask],
])


wake_loss_matrix = load_wake_loss_data(
    f"../../results/precomputed/wake_loss_{GRID_RESOLUTION}m.npz"
)["wake_loss_matrix"]

In [ ]:
LAYOUT_PATH = "../../results/layouts/gurobi/gurobi_128_3600s.yaml"
result = bitswap_postprocess(
    layout_yaml=LAYOUT_PATH,
    candidate_locations=candidate_locations,
    wake_loss_matrix=wake_loss_matrix,
    radius_m=128,
    top_per_turbine=5,
    top_global=10,
    max_iterations=10
)

NameError: name 'compute_aep_from_coords' is not defined